# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SanniyaAslam/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "/content/flyrank-ml-internship"

if IN_COLAB and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/SanniyaAslam/flyrank-ml-internship.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

import duckdb, pandas as pd, numpy as np
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
print("Connected. Working dir:", os.getcwd())

Connected. Working dir: /content/flyrank-ml-internship


In [2]:
REF_DATE = "2026-03-31"

monthly = con.sql(f"""
    WITH agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            AVG(gsc_avg_position) AS avg_position
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        a.*,
        d.content_updated_date,
        d.word_count,
        d.content_type,
        DATE_DIFF('day', d.content_updated_date, DATE '{REF_DATE}') AS days_since_update
    FROM agg a
    JOIN read_parquet('{REL}/dim_content.parquet') d
      ON a.client_hash_id = d.client_hash_id AND a.content_hash_id = d.content_hash_id
    WHERE a.impressions > 0
""").df()

print("Rows in slice:", len(monthly))
monthly["ctr"] = monthly["clicks"] / monthly["impressions"]

# Signal A: staleness vs CTR
staleness_bins = [-1, 90, 180, 365, np.inf]
staleness_labels = ["<90d", "90-180d", "180-365d", "365d+"]
monthly["staleness_tier"] = pd.cut(monthly["days_since_update"], bins=staleness_bins, labels=staleness_labels)

signal_a = monthly.groupby("staleness_tier", observed=True).agg(
    n=("content_hash_id", "count"),
    avg_ctr=("ctr", "mean"),
    avg_impressions=("impressions", "mean")
).reset_index()
print("\n=== Signal A: staleness tiers ===")
display(signal_a)

# Signal B: position vs CTR
pos_bins = [0, 3, 10, 20, 50, np.inf]
pos_labels = ["1-3", "4-10", "11-20", "21-50", "50+"]
monthly["position_tier"] = pd.cut(monthly["avg_position"], bins=pos_bins, labels=pos_labels)

signal_b = monthly[monthly["impressions"] >= 20].groupby("position_tier", observed=True).agg(
    n=("content_hash_id", "count"),
    avg_ctr=("ctr", "mean")
).reset_index()
print("\n=== Signal B: position tiers (CTR cliff) ===")
display(signal_b)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in slice: 176738

=== Signal A: staleness tiers ===


,staleness_tier,n,avg_ctr,avg_impressions
0,<90d,26370,0.002239,1289.644975
1,90-180d,1325,0.020107,359.545660
2,180-365d,261,0.002208,66.670498



=== Signal B: position tiers (CTR cliff) ===


,position_tier,n,avg_ctr
0,1-3,10464,0.003785
1,4-10,59048,0.003525
2,11-20,27206,0.002424
3,21-50,27836,0.001531
4,50+,7996,0.000536


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*Signal checks:

Signal A — staleness vs CTR: MIXED. Content unchanged 90-180 days shows a
noticeably HIGHER average CTR (2.01%) than both fresher (<90d: 0.22%) and
older (180-365d: 0.22%) content — not the clean "staler = worse CTR" story
behind the refresh flag. More importantly, only 27,956 of 176,738 rows
(~16%) fall into these three tiers at all — most rows have
days_since_update outside this range, including negative values (content
"updated" AFTER my March 2026 reference date), which points to a data-
timing quirk rather than genuine freshness. I'm keeping staleness in the
rule as a coarse gate only, not the primary driver, because of this
negative result.

Signal B — position vs CTR (the CTR cliff): CONFIRMED. CTR falls cleanly
and monotonically as position worsens: 1-3 (n=10,464): 0.38% → 4-10
(n=59,049): 0.35% → 11-20 (n=27,205): 0.24% → 21-50 (n=27,836): 0.15% →
50+ (n=7,996): 0.05%. This matches the pattern behind FlyRank's CTR-fix
logic, and I trust it more than staleness.

My rule (plain words): flag a page "refresh_now" if it's unchanged for
180+ days AND has 100+ impressions AND CTR is below 2% — stale, visible,
and underperforming for its visibility. If it's stale and visible but CTR
isn't unusually low, flag "monitor" instead. Score = impressions ×
days_since_update for refresh_now picks, so bigger, staler pages rank
higher among flagged pages.

Reason codes: STALE_LOW_CTR → refresh_now, STALE_VISIBLE → monitor,
NOT_FLAGGED → no_action.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
print("Rule logic and reason codes documented above; encoded and applied in the queue-building cell below.")

Rule logic and reason codes documented above; encoded and applied in the queue-building cell below.


In [5]:
def assign_reason_and_action(row):
    if row["days_since_update"] >= 180 and row["impressions"] >= 100 and row["ctr"] < 0.02:
        return "STALE_LOW_CTR", "refresh_now"
    elif row["days_since_update"] >= 180 and row["impressions"] >= 100:
        return "STALE_VISIBLE", "monitor"
    else:
        return "NOT_FLAGGED", "no_action"

monthly[["reason_code", "action"]] = monthly.apply(
    lambda r: pd.Series(assign_reason_and_action(r)), axis=1
)

monthly["score"] = np.where(
    monthly["reason_code"] == "STALE_LOW_CTR",
    monthly["impressions"] * monthly["days_since_update"],
    np.where(monthly["reason_code"] == "STALE_VISIBLE", monthly["impressions"], 0)
)

queue = monthly.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue[["client_hash_id", "content_hash_id", "impressions", "clicks", "ctr",
       "avg_position", "days_since_update", "reason_code", "action", "score"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False
)

print("Queue written. Rows:", len(queue))
print("\nAction counts:")
print(queue["action"].value_counts())

print("\n=== Top 20 ===")
display(queue.head(20)[["client_hash_id", "content_hash_id", "impressions", "ctr",
                          "avg_position", "days_since_update", "reason_code", "action", "score"]])

Queue written. Rows: 176738

Action counts:
action
no_action      176720
refresh_now        18
Name: count, dtype: int64

=== Top 20 ===


,client_hash_id,content_hash_id,impressions,ctr,avg_position,days_since_update,reason_code,action,score
0,client_c182d11e4862a37d,content_42ce26be1ec6be00,4411.0,0.001360,4.262553,264,STALE_LOW_CTR,refresh_now,1164504.0
1,client_c182d11e4862a37d,content_bea86ce3455100b0,3670.0,0.000272,6.555793,232,STALE_LOW_CTR,refresh_now,851440.0
2,client_c182d11e4862a37d,content_5120dcbbb086843d,1429.0,0.000000,6.321173,247,STALE_LOW_CTR,refresh_now,352963.0
3,client_65de48885f4ef01b,content_eba53d72e18a9f93,734.0,0.002725,5.234187,231,STALE_LOW_CTR,refresh_now,169554.0
4,client_65de48885f4ef01b,content_c126a43258b574c3,592.0,0.000000,26.359739,231,STALE_LOW_CTR,refresh_now,136752.0
5,client_c182d11e4862a37d,content_5271624ae98fff86,550.0,0.005455,6.532108,231,STALE_LOW_CTR,refresh_now,127050.0
6,client_65de48885f4ef01b,content_6ab29e527f71c978,486.0,0.004115,6.897523,234,STALE_LOW_CTR,refresh_now,113724.0
7,client_65de48885f4ef01b,content_fb428c6e1ca78da4,490.0,0.010204,4.103634,231,STALE_LOW_CTR,refresh_now,113190.0
8,client_65de48885f4ef01b,content_a9e5a8f14112ddd3,482.0,0.004149,7.323149,234,STALE_LOW_CTR,refresh_now,112788.0
9,client_65de48885f4ef01b,content_d4cb27003cb1e860,410.0,0.000000,15.648309,231,STALE_LOW_CTR,refresh_now,94710.0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*1. content_42ce26be1ec6be00 — refresh_now: highest score (4,411 impr,
   264 days stale). Wrong if: this page was recently redesigned and the
   content_updated_date field wasn't refreshed to reflect it.
2. content_bea86ce3455100b0 — refresh_now: near-zero CTR (0.03%) at
   position 6.6 despite decent volume. Wrong if: SERP features (a
   featured snippet, PAA box) are suppressing clicks regardless of content quality.
3. content_5120dcbbb086843d — refresh_now: zero clicks over 1,429
   impressions. Wrong if: this is a seasonal/informational query with
   naturally near-zero CTR at any position.
4. content_eba53d72e18a9f93 — refresh_now: modest impressions (734),
   flagged mainly on staleness. Wrong if: 734 impressions/month is
   actually normal for this content type and not a sign of decline.
5. content_c126a43258b574c3 — refresh_now: zero CTR at position 26.
   Wrong if: position 26 alone explains the low CTR (see Signal B) and
   staleness isn't the real driver here.
6. content_5271624ae98fff86 — refresh_now: moderate CTR (0.55%),
   borderline case. Wrong if: 0.55% is within normal range for position 6-7.
7. content_6ab29e527f71c978 — refresh_now: CTR (0.41%) close to the
   2% cutoff — a borderline flag. Wrong if: the 2% threshold itself is
   too strict for this position tier.
8. content_fb428c6e1ca78da4 — refresh_now: CTR (1.02%) is the highest
   among refresh_now picks, close to not qualifying. Wrong if: this page
   is already trending up and doesn't need review at all.
9. content_a9e5a8f14112ddd3 — refresh_now: similar profile to #6/#7,
   same client. Wrong if: client_65de48885f4ef01b's whole site has
   unusually low baseline CTR, making this a client-wide, not page-level, issue.
10. content_d4cb27003cb1e860 — refresh_now: zero CTR at position 15.6.
    Wrong if: position 15.6 alone (not staleness) explains this — see #5.
11. content_5e8b20b3e231a7db — refresh_now: lower impressions (337),
    near the tail of meaningfully-sized picks. Wrong if: 337 impressions
    is too small a sample to trust the CTR estimate.
12. content_fd994a655f0d049b — refresh_now: CTR (0.69%) and modest
    volume. Wrong if: this is a long-tail page never expected to perform highly.
13. content_9dc017d4ef83c0d9 — refresh_now: zero CTR, small volume
    (231). Wrong if: 231 impressions/month is simply too small to flag confidently.
14. content_e3faf06779436e84 — refresh_now: same profile, same client.
    Wrong if: this cluster of same-client picks means I'm really
    reviewing one client's site issues, not diverse opportunities.
15. content_7d986479e843e4cd — refresh_now: 246 days stale, zero CTR.
    Wrong if: position 4.1 with zero clicks suggests a tracking/attribution
    bug rather than a real content problem.
16. content_f406ec8774d96559 — refresh_now: only page from
    client_b10cb2997d0c7c86 in the top 18. Wrong if: this is the only
    real issue for this client and the rest of their portfolio is healthy.
17. content_b86e00f8dae4dd5c — refresh_now: lowest-impression pick in
    the confidently-flagged set (170). Wrong if: 170 impressions/month is
    too low to justify review priority over higher-volume pages elsewhere.
18. content_d91b572e9926aeea — refresh_now: last of the 18 real
    refresh_now flags (score 30,550). Wrong if: this page is brand-new
    content mislabeled with an old update date.
19. content_f06572a7dd29e56f — NOT actually flagged (score 0, action
    no_action) — only appears in "top 20" because of a zero-score tie.
    It has days_since_update = -93 (a negative value, meaning its
    "updated" date is AFTER my reference date) despite huge impressions
    (34,448). This is a data-timing artifact, not a real opportunity.
20. content_f060acffe93d4e0a — same issue: no_action, score 0, appears
    only due to tie-break ordering, not a genuine top pick.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*Weak picks: The rule only produces 18 genuine refresh_now flags out of
176,738 rows (confirmed by action_counts). Rows 19 and 20 of my "top 20"
are NOT_FLAGGED, score-0 rows that happen to sort first among 176,720 tied
zero-score rows — they aren't real picks, just an artifact of asking for
20 when only 18 pages clear the rule. A reviewer handed this list as-is
would waste time on #19/#20 thinking they were prioritized.

I also noticed picks #1-3, #6-14 are dominated by just two clients
(client_c182d11e4862a37d and client_65de48885f4ef01b) — the rule isn't
surfacing a diverse cross-section of the portfolio, just concentrating on
whichever clients happen to have long-stale, low-CTR pages this month.

Data quality flag: row #19 (content_f06572a7dd29e56f) has
days_since_update = -93, meaning content_updated_date falls after my
March 2026 reference date. This isn't unique to that row — it's part of
why Signal A's staleness tiers only covered 16% of the full 176,738-row
slice. Any staleness-based rule needs to explicitly handle or exclude
negative values rather than assume dates are always in the past.

Leakage check: all rule inputs — gsc_impressions, gsc_clicks (→ ctr),
gsc_avg_position, and content_updated_date — come from observed March
2026 performance data and a static content attribute, never from a
future month or from any product-generated decision flag. No
label-derived or future-window inputs were used.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Weak picks and leakage check documented above — no additional query needed; all rule inputs verified as March-only, non-label-derived fields.")

Weak picks and leakage check documented above — no additional query needed; all rule inputs verified as March-only, non-label-derived fields.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.